In [4]:
import pandas as pd
import numpy as np
import json
import os
import time
from openai import OpenAI
from dotenv import load_dotenv
import warnings

# 경고 메시지 무시 및 환경변수 로드
warnings.filterwarnings('ignore')
load_dotenv()

print("=" * 80)
print("Step 6: Agent #2 - 재무 개선 컨설팅 보고서 생성 (최종 전체 실행)")
print("=" * 80)

# ============================================================================
# 1. 변수명 매핑 사전 (Feature Mapping)
# ============================================================================
FEATURE_MAPPING = {
    'FN1_1': '유동자산', 'FN1_2': '비유동자산', 'FN1_3': '당좌자산', 
    'FN1_4': '재고자산', 'FN1_5': '유형자산', 'FN1_6': '재공품',
    'FN1_7': '현금', 'FN1_8': '현금등가물', 'FN1_9': '상품유가증권',
    'FN1_10': '현금성자산', 'FN1_11': '매출채권', 'FN1_11_1': '전기매출채권',
    'FN1_11_2': '매출채권처분손실', 'FN1_11_3': '무형자산', 'FN1_11_4': '투자자산',
    'FN1_13': '자산총계', 'FN1_13_1': '전기자산총계', 'FN1_14': '유동부채',
    'FN1_15': '단기차입금', 'FN1_16': '차입금', 'FN1_17': '매입채무',
    'FN1_18': '비유동부채', 'FN1_19': '부채총계', 'FN1_20': '자기자본',
    'FN1_21': '자본잉여금', 'FN1_22': '이익잉여금', 'FN1_23': '유보금',
    'FN1_24': '자본총계', 'FN2_1': '매출액', 'FN2_1_1': '전기매출액',
    'FN2_2': '매출원가', 'FN2_2_1': '매출총이익', 'FN2_3': '판관비',
    'FN2_4': '금융비용', 'FN2_5': '영업손익', 'FN2_5_1': '전기영업이익',
    'FN2_7': '영업외수익', 'FN2_8': '영업외비용', 'FN2_9': '법인세차감전순이익',
    'FN2_10': '당기순이익', 'FN2_10_1': '전기당기순이익',
    'FN3_1': '현금흐름', 'FN3_2': '영업활동현금흐름', 'FN3_2_1': '투자활동현금흐름',
    'FN3_3': '부채상환계수', 'FN3_6': '적립금비율', 'FN3_7': 'EBIT',
    'FN3_8': 'EBITDA', 'FN3_10': '청산가치율', 'FN3_11': '순운전자본',
    'FN3_11_1': '순차입금', 'PERF_12M': '부도여부'
}

# ============================================================================
# 2. 데이터 로드
# ============================================================================
try:
    df_input = pd.read_csv('agent1_interpretation_results.csv')
    print(f"데이터 로드 완료: {len(df_input)}건")
except FileNotFoundError:
    raise FileNotFoundError("이전 단계 파일(agent1_interpretation_results.csv)이 없습니다.")

# ============================================================================
# 3. Report Generator Class (논리 강화 버전)
# ============================================================================
class ReportGenerator:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = os.getenv('LLM_MODEL', 'gpt-4o-mini')

    def _format_financial_guide(self, row):
        """수치 변화 가이드 및 해석 힌트 생성"""
        changes_list = []
        features = [col.replace('Original_', '') for col in row.index if col.startswith('Original_')]
        
        for feat in features:
            try:
                original = float(row[f'Original_{feat}'])
                target = float(row[f'CF_{feat}'])
                change = float(row[f'Change_{feat}'])
                
                if abs(change) > 0.001: 
                    pct_change = (change / abs(original)) * 100 if original != 0 else 0
                    direction = "증가" if change > 0 else "감소"
                    feat_name = FEATURE_MAPPING.get(feat, feat)
                    
                    # [Logic Guardrail] 변수 방향성에 따른 해석 힌트
                    hint = ""
                    if "차입금" in feat_name or "부채" in feat_name:
                        if direction == "증가":
                            hint = " -> (주의: 유동성/운영자금 확보를 위한 전략적 차입 확대임. '부채 상환' 금지)"
                        else:
                            hint = " -> (재무 건전성 강화를 위한 상환)"
                    elif "자산" in feat_name or "재고" in feat_name:
                        if direction == "감소":
                            hint = " -> (유휴 자산 매각 또는 재고 처분)"
                        else:
                            hint = " -> (기업 규모 확대 및 자산 확충)"
                    elif "매출채권" in feat_name:
                        if direction == "감소":
                            hint = " -> (적극적인 채권 회수를 통한 현금화)"

                    # 포맷: "- 매출채권: 100 -> 50 (50% 감소) -> (힌트)"
                    desc = f"- **{feat_name}**: {original:,.0f} → {target:,.0f} ({abs(pct_change):.1f}% {direction}){hint}"
                    changes_list.append((abs(pct_change), desc))
            except Exception:
                continue
        
        # 변화율 큰 순서대로 상위 10개
        changes_list.sort(key=lambda x: x[0], reverse=True)
        return "\n".join([item[1] for item in changes_list[:10]])

    def generate_report(self, row):
        company_id = row['ID']
        financial_guide_text = self._format_financial_guide(row)
        prob_improvement = (row['Original_Proba'] - row['Target_Proba']) * 100
        
        system_prompt = """
        당신은 기업신용보험 언더라이팅 전문가입니다. 
        숫자의 증가/감소 방향을 정확히 해석하여 논리적인 재무 개선 보고서를 작성하세요.
        
        [절대 원칙]
        1. **데이터 기반 해석:** 데이터가 '부채 증가'를 지시하면 '자금 조달'로 해석하고, 절대 '부채 상환'이라고 쓰지 마세요.
        2. **용어 사용:** 변수 코드(FN...) 대신 한글 용어를 사용하세요.
        3. **구체성:** [재무 지표 변화 가이드]의 수치를 본문에 반드시 인용하세요.
        """
        
        user_prompt = f"""
        [기업 정보]
        - ID: {company_id}
        - 부도 확률 개선: {row['Original_Proba']:.1%} → {row['Target_Proba']:.1%} ({prob_improvement:.1f}%p 개선 목표)
        
        [AI 분석 요약]
        - 전략: {row.get('selection_reason', '-')}
        - 현실성: {row.get('feasibility_assessment', '-')}
        
        [재무 지표 변화 가이드 (수치 및 해석 힌트)]
        {financial_guide_text}
        
        [보고서 작성 (Markdown)]
        # 1. Executive Summary (3문장 요약)
        # 2. Current Status Analysis (현황 진단)
        # 3. Improvement Scenarios (상세 목표 - 수치 인용 필수)
        # 4. Action Roadmap (단기/중기)
        # 5. Risk Management
        # 6. Performance KPIs (3가지)
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0.5 
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

# ============================================================================
# 4. 전체 실행 (Full Run)
# ============================================================================
generator = ReportGenerator()
generated_reports = []

# [설정] TEST_MODE = False (전체 실행)
TEST_MODE = False 

print(f"\n[FULL MODE] 전체 {len(df_input)}개 기업에 대해 보고서를 생성합니다.")
print("진행 중... (약 10~20분 소요 예상)")

# 실행 루프
try:
    from tqdm import tqdm
    iterator = tqdm(df_input.iterrows(), total=len(df_input), desc="Generating Reports")
except ImportError:
    iterator = df_input.iterrows()

for idx, row in iterator:
    report_text = generator.generate_report(row)
    generated_reports.append({
        'company_id': row['ID'],
        'original_proba': row['Original_Proba'],
        'target_proba': row['Target_Proba'],
        'report_content': report_text
    })

# ============================================================================
# 5. 저장
# ============================================================================
output_csv = 'agent2_consulting_reports.csv'
df_reports = pd.DataFrame(generated_reports)
df_reports.to_csv(output_csv, index=False, encoding='utf-8-sig')

print("\n" + "="*80)
print(f"생성 완료: 총 {len(df_reports)}건 저장됨")
print(f"통합 파일: {output_csv}")
print("="*80)

# 개별 Markdown 파일 저장 (reports 폴더)
output_dir = 'reports'
os.makedirs(output_dir, exist_ok=True)
for _, row in df_reports.iterrows():
    with open(f"{output_dir}/Report_{int(row['company_id'])}.md", 'w', encoding='utf-8') as f:
        f.write(row['report_content'])

print(f"개별 보고서 저장 완료: {output_dir}/ 폴더 내")

Step 6: Agent #2 - 재무 개선 컨설팅 보고서 생성 (최종 전체 실행)
데이터 로드 완료: 266건

[FULL MODE] 전체 266개 기업에 대해 보고서를 생성합니다.
진행 중... (약 10~20분 소요 예상)


Generating Reports: 100%|██████████████████████████████████████████████████████████| 266/266 [1:03:56<00:00, 14.42s/it]



생성 완료: 총 266건 저장됨
통합 파일: agent2_consulting_reports.csv
개별 보고서 저장 완료: reports/ 폴더 내
